# EDGECAST — Team 29, A2 Slot\nGenerative AI Service Provision in Heterogeneous Edge Networks: Stability-Controlled Cache-Aware Enhancement\n\nThis notebook runs the actual EDGECAST implementation submitted for faculty evaluation.\n\nRepository: priyam1206/edgecast-a2-submission\n\nIncluded:\n- Nguyen et al. reduced Stage-2 comparator;\n- stabilized cache-aware EDGECAST controller;\n- checkpoint/cache state and background staging;\n- warm/cold cutover;\n- minimum allocation persistence;\n- real processed FineServe and BurstGPT traces;\n- automated tests and benchmark scripts.\n\nMethodological boundary: this demonstrator is model-aggregated. The reduced P3 objective is optimized exactly for the small demo state; exact enumeration is used instead of reproducing the base paper's Benders solver line-for-line.\n

## 1. Clone the public submission repository\n

In [ ]:
!rm -rf /content/edgecast\n!git clone -q https://github.com/priyam1206/edgecast-a2-submission.git /content/edgecast\n%cd /content/edgecast\n!git rev-parse --short HEAD\n

## 2. Install dependencies\n

In [ ]:
!python -m pip install -q -r requirements.txt\n

## 3. Verify the implementation\n

In [ ]:
!python -m pytest -q\n

## 4. Real FineServe benchmark — quick verification\nThis runs the actual paper comparator, no-persistence ablation, and stabilized EDGECAST controller on the shipped FineServe trace.\n

In [ ]:
!python scripts/benchmark_fineserve.py \\n  --trace data/preprocessed/FineServe_lite_60s.csv \\n  --windows 5 \\n  --window-slots 120 \\n  --nodes 6 \\n  --seed 42 \\n  --mode requests \\n  --reallocation-persistence 6 \\n  --out results/fineserve_quick.csv\n

In [ ]:
import pandas as pd\nfrom IPython.display import display\n\nfine = pd.read_csv("results/fineserve_quick.csv")\ndisplay(fine)\n\nsummary = pd.DataFrame({\n    "Metric": [\n        "Mean latency change vs paper (%)",\n        "Mean backlog change vs paper (%)",\n        "Mean downtime change vs paper (%)",\n        "Mean paper switches",\n        "Mean no-persistence switches",\n        "Mean stabilized EDGECAST switches",\n    ],\n    "Value": [\n        fine["stable_latency_delta_pct"].mean(),\n        fine["stable_backlog_delta_pct"].mean(),\n        fine["stable_downtime_delta_pct"].mean(),\n        fine["paper_switches"].mean(),\n        fine["legacy_switches"].mean(),\n        fine["stable_switches"].mean(),\n    ],\n})\ndisplay(summary)\n

In [ ]:
import matplotlib.pyplot as plt\nax = fine[["paper_avg_latency","stable_avg_latency"]].plot(\n    kind="bar", figsize=(10,4), title="FineServe: Average latency per sampled window"\n)\nax.set_xlabel("Window")\nax.set_ylabel("Average latency (slots)")\nplt.tight_layout()\nplt.show()\n

## 5. Full 30-window FineServe reproducibility run\n

In [ ]:
!python scripts/benchmark_fineserve.py \\n  --trace data/preprocessed/FineServe_lite_60s.csv \\n  --windows 30 \\n  --window-slots 120 \\n  --nodes 6 \\n  --seed 42 \\n  --mode requests \\n  --reallocation-persistence 6 \\n  --out results/fineserve_30_windows.csv\n

## 6. Full 30-window BurstGPT reproducibility run\n

In [ ]:
!python scripts/benchmark_burstgpt.py \\n  --trace data/preprocessed/BurstGPT_1_60s.csv \\n  --windows 30 \\n  --window-slots 180 \\n  --nodes 6 \\n  --seed 42 \\n  --mode requests \\n  --reallocation-persistence 6 \\n  --out results/burstgpt_30_windows.csv\n

In [ ]:
burst = pd.read_csv("results/burstgpt_30_windows.csv")\ndisplay(burst.head())\nprint("Mean latency change vs paper: %.2f%%" % burst["avg_latency_delta_pct"].mean())\nprint("Mean backlog change vs paper: %.2f%%" % burst["avg_backlog_delta_pct"].mean())\nprint("Mean paper switches: %.2f" % burst["paper_switches"].mean())\nprint("Mean EDGECAST switches: %.2f" % burst["edgecast_switches"].mean())\n

## 7. Live deployed demo\n\nhttps://edgecast-demo.onrender.com\n\nThe deployed Streamlit app executes the same simulator/controller server-side and exposes live run verification and downloadable traces.\n